# KKBox Churn Prediction — Step 3: Modeling
**File:** `03_Modeling.ipynb`
**Inputs:** `master_model_table.parquet`, `inference_snapshot.parquet` from `02_FeatureEngineering.ipynb`
**Outputs:** `submission.csv`, model files, `modeling_metadata.json`

---

This notebook trains three models, compares results, builds a weighted ensemble, and generates predictions for the March 2017 inference population.

## Results (current run — LGBM best_iter=6 due to num_uniq bug in FE)

These numbers are from the run **before** the FE `num_uniq` fix. After re-running
`02_FeatureEngineering.ipynb` with the fix and retraining, expect LGBM to converge
properly and LogLoss to improve significantly.

| Model | Val AUC | Val LogLoss | Time |
|---|---|---|---|
| Logistic Regression | 0.702 | 0.607 | 583s |
| LightGBM | 0.761 | 0.298 | 294s |
| XGBoost | 0.768 | 0.545 | 732s |
| Ensemble 12/88 (Bryan) | 0.768 | 0.481 | - |
| Optimal ensemble (95/5) | 0.769 | - | - |

Comparison against competition leaderboard:
- Bryan Gregory (1st place): LogLoss 0.07974
- Bryan XGB without logs (17 features): LogLoss 0.10453
- Current model (53 features): LogLoss 0.298 (LGBM best)

The gap is primarily caused by LGBM stopping at iteration 6 (degenerate early stopping) and log features being all-zero in this run because the `num_uniq` column fix had not yet been applied to the FE output.

## Input data

| Split | Users | Churn | Labels |
|---|---|---|---|
| Train (Jan 2017) | 992,931 | 6.39% | official |
| Validation (Feb 2017) | 970,960 | 8.99% | official |
| Inference (Mar 2017) | 907,471 | none | none |

53 features, 0 NaN, no meta columns in feature matrix. Class imbalance ratio: 14.64:1 (929,460 non-churn / 63,471 churn).


In [18]:
# ===== 1. Imports =====
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.pipeline import Pipeline

# Tree models
import lightgbm as lgb
import xgboost as xgb

# Hyperparameter tuning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.6f}".format)
print("✅ Imports OK")
print(f"  lightgbm : {lgb.__version__}")
print(f"  xgboost  : {xgb.__version__}")
print(f"  optuna   : {optuna.__version__}")


✅ Imports OK
  lightgbm : 4.6.0
  xgboost  : 3.0.5
  optuna   : 4.6.0


In [19]:
# ===== 2. Paths + Load =====
DATA_DIR    = Path("Data")
MODELS_DIR  = Path("Models")
MODELS_DIR.mkdir(exist_ok=True)

MASTER_PATH    = DATA_DIR / "master_model_table.parquet"
INFERENCE_PATH = DATA_DIR / "inference_snapshot.parquet"
FE_META_PATH   = DATA_DIR / "feature_engineering_metadata_v7.json"

for p in [MASTER_PATH, INFERENCE_PATH]:
    status = "FOUND ✅" if p.exists() else "MISSING ❌"
    print(f"{p}: {status}")

master = pd.read_parquet(MASTER_PATH)
inf_df = pd.read_parquet(INFERENCE_PATH)

fe_meta = {}
if FE_META_PATH.exists():
    with open(FE_META_PATH) as f:
        fe_meta = json.load(f)

print(f"\nmaster shape    : {master.shape}")
print(f"inference shape : {inf_df.shape}")
print(f"\nColumns         : {list(master.columns)}")
print(f"\nRows by split:")
print(master.groupby("dataset_split").size())
print(f"\nChurn by split:")
print(master.groupby("dataset_split")["is_churn"].mean())


Data\master_model_table.parquet: FOUND ✅
Data\inference_snapshot.parquet: FOUND ✅

master shape    : (1963891, 60)
inference shape : (907471, 56)

Columns         : ['msno', 'snapshot_date', 'last_expire', 'is_churn', 'label_source', 'dataset_split', 'bd', 'bd_missing', 'city', 'city_missing', 'gender', 'gender_missing', 'registered_via', 'registered_via_missing', 'days_since_reg', 'registration_date_abs', 'n_txns', 'auto_renew_rate', 'last_is_auto_renew', 'avg_plan_days', 'plan_days_std', 'plan_change_flag', 'last_plan_days', 'first_plan_days', 'max_plan_days', 'total_amount_paid', 'avg_amount_paid', 'max_amount_paid', 'zero_paid_rate', 'avg_discount_rate', 'share_30d', 'share_90d', 'days_since_last_txn', 'tenure_days', 'cancel_rate', 'cancel_count', 'last_is_cancel', 'cancel_in_last_month', 'avg_cancel_per_month', 'cancel_to_plandays_ratio', 'active_no_cancel_flag', 'days_since_last_cancel', 'has_ever_cancelled', 'payment_method_nunique', 'days_last_txn_to_expire', 'recency_to_plan_r

In [20]:
# ===== 3. Data Preparation =====

# ── Meta + feature columns ────────────────────────────────────────────────────
META_COLS = ["msno", "snapshot_date", "last_expire",
             "is_churn", "label_source", "dataset_split"]
FEATURE_COLS = [c for c in master.columns if c not in META_COLS]
TARGET = "is_churn"

CAT_COLS = ["city", "gender", "registered_via"]
NUM_COLS = [c for c in FEATURE_COLS if c not in CAT_COLS]

print(f"Feature cols ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"\nCategorical  ({len(CAT_COLS)}): {CAT_COLS}")
print(f"Numeric      ({len(NUM_COLS)}): {NUM_COLS}")

# ── Train / Val split ─────────────────────────────────────────────────────────
train_df = master[master["dataset_split"] == "train"].copy()
val_df   = master[master["dataset_split"] == "validation"].copy()

X_train = train_df[FEATURE_COLS].copy()
y_train = train_df[TARGET].astype(int)
X_val   = val_df[FEATURE_COLS].copy()
y_val   = val_df[TARGET].astype(int)
X_inf   = inf_df[FEATURE_COLS].copy()

print(f"\nX_train : {X_train.shape}  |  y_train churn: {y_train.mean():.4f}")
print(f"X_val   : {X_val.shape}  |  y_val   churn: {y_val.mean():.4f}")
print(f"X_inf   : {X_inf.shape}")

# ── Step 1: Fix string dtype on CAT_COLS before OrdinalEncoder ────────────────
for df in [X_train, X_val, X_inf]:
    for c in CAT_COLS:
        df[c] = df[c].astype(object).fillna("Unknown").astype(str)

# ── Step 2: OrdinalEncoder ────────────────────────────────────────────────────
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
oe.fit(X_train[CAT_COLS])
for df in [X_train, X_val, X_inf]:
    df[CAT_COLS] = oe.transform(df[CAT_COLS]).astype(float)

print("\nOrdinal encoding done ✅")

# ── Step 3: Force ALL feature cols to float64, fill NaN/inf ──────────────────
# Must happen AFTER OrdinalEncoder so CAT_COLS are already numeric.
# Medians computed from train only — applied to val/inf (no leakage).
for df in [X_train, X_val, X_inf]:
    for c in FEATURE_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")

train_medians = X_train.median()  # computed after coerce, before fillna
for df in [X_train, X_val, X_inf]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(train_medians, inplace=True)

# ── Step 4: Final assertion ───────────────────────────────────────────────────
nan_train = X_train.isna().sum().sum()
nan_val   = X_val.isna().sum().sum()
nan_inf   = X_inf.isna().sum().sum()
print(f"NaN after cleanup — X_train: {nan_train}, X_val: {nan_val}, X_inf: {nan_inf}")
assert nan_train == 0 and nan_val == 0, f"NaN remain: train={nan_train}, val={nan_val}"
print("✅ All features clean — ready for modeling")

# ── Class imbalance ratio ─────────────────────────────────────────────────────
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
SCALE_POS_WEIGHT = n_neg / n_pos
print(f"\nClass imbalance — neg: {n_neg:,}  pos: {n_pos:,}  ratio: {SCALE_POS_WEIGHT:.2f}")


Feature cols (54): ['bd', 'bd_missing', 'city', 'city_missing', 'gender', 'gender_missing', 'registered_via', 'registered_via_missing', 'days_since_reg', 'registration_date_abs', 'n_txns', 'auto_renew_rate', 'last_is_auto_renew', 'avg_plan_days', 'plan_days_std', 'plan_change_flag', 'last_plan_days', 'first_plan_days', 'max_plan_days', 'total_amount_paid', 'avg_amount_paid', 'max_amount_paid', 'zero_paid_rate', 'avg_discount_rate', 'share_30d', 'share_90d', 'days_since_last_txn', 'tenure_days', 'cancel_rate', 'cancel_count', 'last_is_cancel', 'cancel_in_last_month', 'avg_cancel_per_month', 'cancel_to_plandays_ratio', 'active_no_cancel_flag', 'days_since_last_cancel', 'has_ever_cancelled', 'payment_method_nunique', 'days_last_txn_to_expire', 'recency_to_plan_ratio', 'spend_rate_per_day', 'total_secs_played', 'num_logins', 'secs_played_14d', 'logins_14d', 'secs_played_30d', 'logins_30d', 'logins_90d', 'secs_played_60_30d', 'delta_secs_30d_vs_prior', 'days_since_last_login', 'max_secs_sin

## Logistic Regression Baseline

### Role in the pipeline

Logistic Regression serves as a baseline to confirm that features carry linear signal. If LR achieves AUC meaningfully above 0.5, the features provide basic predictive value.

### Algorithm

Logistic Regression models churn probability as a sigmoid of a linear combination of features. It is fast and interpretable but cannot learn nonlinear interactions between features.

Settings used:
- `class_weight='balanced'`: automatically adjusts sample weights by class frequency (14.64:1 imbalance), preventing the model from predicting non-churn for all users
- `StandardScaler`: normalizes features to mean=0, std=1, required because LR is sensitive to feature scale
- `solver='saga'`: faster than lbfgs for large datasets, supports L1/L2 regularization
- `max_iter=1000`: sufficient iterations to converge on 1M+ samples

### Results and observations

Val AUC = 0.702: the linear model captures about 70% of the classification signal. This confirms features like `auto_renew_rate`, `days_since_reg`, and `registered_via` carry meaningful linear signal.

Val LogLoss = 0.607 is high because the predicted probabilities are not well calibrated for this imbalanced problem.

Training time of 583s reflects the cost of running saga on ~1M samples with 53 features.


In [21]:
# ===== 4. Logistic Regression Baseline =====
from sklearn.preprocessing import StandardScaler

start_lr = time.time()

# Scale for LR only (tree models don't need this)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

lr = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    C=1.0,
    solver="saga",      # fast for large datasets
    random_state=42,
    n_jobs=-1,
)
lr.fit(X_train_scaled, y_train)

lr_val_proba = lr.predict_proba(X_val_scaled)[:, 1]
lr_auc       = roc_auc_score(y_val, lr_val_proba)
lr_logloss   = log_loss(y_val, lr_val_proba)
lr_gini      = 2 * lr_auc - 1
lr_time      = time.time() - start_lr

print("=" * 55)
print("LOGISTIC REGRESSION — RESULTS")
print("=" * 55)
print(f"  Val AUC      : {lr_auc:.6f}")
print(f"  Val Gini     : {lr_gini:.6f}")
print(f"  Val LogLoss  : {lr_logloss:.6f}")
print(f"  Train time   : {lr_time:.1f}s")
print("=" * 55)

LR_RESULTS = {"model": "Logistic Regression", "auc": lr_auc,
              "gini": lr_gini, "logloss": lr_logloss, "time": lr_time}


LOGISTIC REGRESSION — RESULTS
  Val AUC      : 0.701957
  Val Gini     : 0.403915
  Val LogLoss  : 0.605552
  Train time   : 616.1s


## LightGBM with Optuna Tuning

### Role in the pipeline

LightGBM is a gradient boosting model based on decision trees. In Bryan Gregory's ensemble, LGBM contributes 12% of the final prediction. It is well-suited to categorical features and sparse data.

### Algorithm

Gradient boosting builds trees sequentially, each correcting the errors of the previous one. LightGBM uses leaf-wise tree growth (versus level-wise in XGBoost), which achieves greater depth with fewer leaves but is more prone to overfitting without proper regularization.

Key hyperparameters searched by Optuna:
- `num_leaves`: maximum leaves per tree (20-300), controls model complexity
- `learning_rate`: step size per round (0.001-0.3), smaller values are more stable but need more rounds
- `feature_fraction`, `bagging_fraction`: fraction of features/data sampled per round, prevents overfitting
- `lambda_l1`, `lambda_l2`: regularization strength
- `is_unbalance=True`: handles class imbalance automatically within LGBM

### Optuna tuning

Optuna's TPE sampler (Tree-structured Parzen Estimator) builds a probabilistic model over hyperparameter space and focuses search on regions that have historically produced high AUC, making it more efficient than random or grid search.

100 trials used (increased from 50 because best trial was 81/100, indicating convergence was not reached).

### Results and key issue

Val AUC = 0.761, Val LogLoss = 0.298 (best single model for LogLoss).

Critical issue: `best_iteration = 6`. LGBM stopped training after 6 boosting rounds because early stopping (patience=50) detected no improvement in validation AUC. Two likely causes:
1. `num_leaves=173` is too large relative to the effective signal available — with log features all-zero, the model overfits quickly
2. Log features being all-zero (run before `num_uniq` fix) — the model exhausts useful patterns very early

This will change significantly once log features carry real data.


In [22]:
# ===== 5. LightGBM + Optuna =====
N_TRIALS_LGBM = 100  # increased from 50 — best was trial 21/50, not converged

def lgbm_objective(trial):
    params = {
        "objective"          : "binary",
        "metric"             : "auc",
        "verbosity"          : -1,
        "boosting_type"      : "gbdt",
        "random_state"       : 42,
        "n_jobs"             : -1,
        "is_unbalance"       : True,
        "num_leaves"         : trial.suggest_int("num_leaves", 20, 300),
        "learning_rate"      : trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "n_estimators"       : trial.suggest_int("n_estimators", 100, 1000),
        "min_child_samples"  : trial.suggest_int("min_child_samples", 10, 200),
        "feature_fraction"   : trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction"   : trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq"       : 1,
        "lambda_l1"          : trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2"          : trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "max_depth"          : trial.suggest_int("max_depth", 3, 12),
        "min_split_gain"     : trial.suggest_float("min_split_gain", 0.0, 1.0),
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(-1)],
    )
    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

print(f"Starting LGBM Optuna ({N_TRIALS_LGBM} trials)...")
start_lgbm_tune = time.time()

lgbm_study = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42))
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS_LGBM, show_progress_bar=True)

lgbm_tune_time = time.time() - start_lgbm_tune
print(f"\nLGBM tuning done in {lgbm_tune_time:.0f}s")
print(f"Best trial AUC: {lgbm_study.best_value:.6f}")
print(f"Best params   : {lgbm_study.best_params}")

# ── Retrain with best params ──────────────────────────────────────────────────
best_lgbm_params = {
    "objective": "binary", "metric": "auc", "verbosity": -1,
    "boosting_type": "gbdt", "random_state": 42, "n_jobs": -1,
    "is_unbalance": True, **lgbm_study.best_params
}

start_lgbm_fit = time.time()
lgbm_model = lgb.LGBMClassifier(**best_lgbm_params)
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
)

lgbm_val_proba = lgbm_model.predict_proba(X_val)[:, 1]
lgbm_auc       = roc_auc_score(y_val, lgbm_val_proba)
lgbm_logloss   = log_loss(y_val, lgbm_val_proba)
lgbm_gini      = 2 * lgbm_auc - 1
lgbm_time      = time.time() - start_lgbm_fit

print("\n" + "=" * 55)
print("LIGHTGBM — FINAL RESULTS")
print("=" * 55)
print(f"  Val AUC      : {lgbm_auc:.6f}")
print(f"  Val Gini     : {lgbm_gini:.6f}")
print(f"  Val LogLoss  : {lgbm_logloss:.6f}")
print(f"  Best iter    : {lgbm_model.best_iteration_}")
print(f"  Fit time     : {lgbm_time:.1f}s")
print("=" * 55)

LGBM_RESULTS = {"model": "LightGBM", "auc": lgbm_auc,
                "gini": lgbm_gini, "logloss": lgbm_logloss,
                "time": lgbm_tune_time + lgbm_time}


Starting LGBM Optuna (100 trials)...


Best trial: 98. Best value: 0.767592: 100%|██████████| 100/100 [06:26<00:00,  3.86s/it]



LGBM tuning done in 386s
Best trial AUC: 0.767592
Best params   : {'num_leaves': 20, 'learning_rate': 0.015534465205188398, 'n_estimators': 214, 'min_child_samples': 65, 'feature_fraction': 0.5517390506108937, 'bagging_fraction': 0.5931908750426373, 'lambda_l1': 5.09309314251787e-07, 'lambda_l2': 0.05873202239720797, 'max_depth': 7, 'min_split_gain': 0.11109909526281778}

LIGHTGBM — FINAL RESULTS
  Val AUC      : 0.765946
  Val Gini     : 0.531892
  Val LogLoss  : 0.290374
  Best iter    : 5
  Fit time     : 2.4s


## XGBoost with Optuna Tuning

### Role in the pipeline

XGBoost is the primary model in Bryan's ensemble (88%). XGBoost uses level-wise tree growth, which is more stable than LightGBM's leaf-wise approach but slower. With 53 features and ~1M rows, `tree_method='hist'` is critical for acceptable training speed.

### Comparison with LightGBM

| Aspect | LightGBM | XGBoost |
|---|---|---|
| Tree growth | Leaf-wise (deeper) | Level-wise (wider) |
| Speed | Faster | Slower |
| Categoricals | Native support | Requires encoding |
| Overfitting risk | Higher | Lower |
| Generalization | Moderate | Better |

Settings used:
- `scale_pos_weight = 14.64`: handles class imbalance for XGBoost
- `tree_method='hist'`: histogram-based algorithm, significantly faster on large datasets with many features (Bryan explicitly named this setting)
- `early_stopping_rounds=50`: stops if validation AUC does not improve for 50 consecutive rounds

### Results

Val AUC = 0.768, Val LogLoss = 0.545.

`best_iteration = 29` is also low, for the same underlying reason as LGBM — log features are all-zero, limiting what the model can learn.

XGBoost has higher AUC than LGBM (0.768 vs 0.761) but worse LogLoss (0.545 vs 0.298). This means XGBoost ranks users better but its predicted probabilities are less calibrated. Since the competition evaluates on LogLoss, LGBM has the advantage here.


In [23]:
# ===== 6. XGBoost + Optuna =====
N_TRIALS_XGB = 100   # increased from 50 — best was trial 40/50, nearly converged

def xgb_objective(trial):
    params = {
        "objective"          : "binary:logistic",
        "eval_metric"        : "auc",
        "random_state"       : 42,
        "n_jobs"             : -1,
        "verbosity"          : 0,
        "tree_method"        : "hist",   # Bryan Slide 5: fast histogram grower
        "scale_pos_weight"   : SCALE_POS_WEIGHT,
        "n_estimators"       : trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate"      : trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth"          : trial.suggest_int("max_depth", 3, 10),
        "min_child_weight"   : trial.suggest_int("min_child_weight", 1, 50),
        "subsample"          : trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree"   : trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "colsample_bylevel"  : trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        "reg_alpha"          : trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda"         : trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "gamma"              : trial.suggest_float("gamma", 0.0, 5.0),
    }
    model = xgb.XGBClassifier(**params, early_stopping_rounds=30,
                               enable_categorical=False)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)], verbose=False)
    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

print(f"Starting XGB Optuna ({N_TRIALS_XGB} trials)...")
start_xgb_tune = time.time()

xgb_study = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS_XGB, show_progress_bar=True)

xgb_tune_time = time.time() - start_xgb_tune
print(f"\nXGB tuning done in {xgb_tune_time:.0f}s")
print(f"Best trial AUC: {xgb_study.best_value:.6f}")
print(f"Best params   : {xgb_study.best_params}")

# ── Retrain with best params ──────────────────────────────────────────────────
best_xgb_params = {
    "objective": "binary:logistic", "eval_metric": "auc",
    "random_state": 42, "n_jobs": -1, "verbosity": 0,
    "tree_method": "hist",
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "early_stopping_rounds": 50,
    "enable_categorical": False,
    **xgb_study.best_params
}

start_xgb_fit = time.time()
xgb_model = xgb.XGBClassifier(**best_xgb_params)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)], verbose=False)

xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]
xgb_auc       = roc_auc_score(y_val, xgb_val_proba)
xgb_logloss   = log_loss(y_val, xgb_val_proba)
xgb_gini      = 2 * xgb_auc - 1
xgb_time      = time.time() - start_xgb_fit

print("\n" + "=" * 55)
print("XGBOOST — FINAL RESULTS")
print("=" * 55)
print(f"  Val AUC      : {xgb_auc:.6f}")
print(f"  Val Gini     : {xgb_gini:.6f}")
print(f"  Val LogLoss  : {xgb_logloss:.6f}")
print(f"  Best iter    : {xgb_model.best_iteration}")
print(f"  Fit time     : {xgb_time:.1f}s")
print("=" * 55)

XGB_RESULTS = {"model": "XGBoost", "auc": xgb_auc,
               "gini": xgb_gini, "logloss": xgb_logloss,
               "time": xgb_tune_time + xgb_time}


Starting XGB Optuna (100 trials)...


Best trial: 91. Best value: 0.765593: 100%|██████████| 100/100 [16:34<00:00,  9.94s/it]



XGB tuning done in 994s
Best trial AUC: 0.765593
Best params   : {'n_estimators': 594, 'learning_rate': 0.054016467945634104, 'max_depth': 3, 'min_child_weight': 47, 'subsample': 0.6162394530110226, 'colsample_bytree': 0.40184331726499695, 'colsample_bylevel': 0.67480798341762, 'reg_alpha': 1.0887592395518634e-08, 'reg_lambda': 1.5512063635287625e-08, 'gamma': 4.470204508344227}

XGBOOST — FINAL RESULTS
  Val AUC      : 0.765593
  Val Gini     : 0.531185
  Val LogLoss  : 0.551099
  Best iter    : 149
  Fit time     : 19.2s


## Model Comparison

Comparison of AUC, Gini, and LogLoss on the validation set (February 2017). AUC is the primary competition metric.

Gini = 2 * AUC - 1, commonly used in financial and subscription industries. Current AUC of 0.769 corresponds to Gini of 0.538.

Distance from competition benchmarks:
- Bryan without logs (17 features): LogLoss 0.105
- Bryan full model (76 features): LogLoss 0.080
- Current model: LogLoss 0.298

The largest gap comes from: (1) log features being all-zero in this run, (2) LGBM stopping at iteration 6, and (3) roughly 23 features fewer than Bryan's final model.


In [24]:
# ===== 7. Model Comparison =====
results = pd.DataFrame([LR_RESULTS, LGBM_RESULTS, XGB_RESULTS])
results = results.sort_values("auc", ascending=False).reset_index(drop=True)
results["rank"] = results.index + 1

print("=" * 65)
print("MODEL COMPARISON — Validation Set (Feb 2017)")
print("=" * 65)
print(results[["rank","model","auc","gini","logloss","time"]].to_string(index=False))
print("=" * 65)
print(f"\nBest model: {results.iloc[0]['model']} (AUC = {results.iloc[0]['auc']:.6f})")
print(f"LR baseline AUC : {LR_RESULTS['auc']:.6f}")
print(f"LGBM gain vs LR : +{LGBM_RESULTS['auc'] - LR_RESULTS['auc']:.6f}")
print(f"XGB  gain vs LR : +{XGB_RESULTS['auc']  - LR_RESULTS['auc']:.6f}")

display(results[["model","auc","gini","logloss"]].style.background_gradient(
    subset=["auc","gini"], cmap="Greens"
).background_gradient(subset=["logloss"], cmap="Reds_r").format({
    "auc": "{:.6f}", "gini": "{:.6f}", "logloss": "{:.6f}"
}))


MODEL COMPARISON — Validation Set (Feb 2017)
 rank               model      auc     gini  logloss        time
    1            LightGBM 0.765946 0.531892 0.290374  388.757625
    2             XGBoost 0.765593 0.531185 0.551099 1013.695168
    3 Logistic Regression 0.701957 0.403915 0.605552  616.068919

Best model: LightGBM (AUC = 0.765946)
LR baseline AUC : 0.701957
LGBM gain vs LR : +0.063989
XGB  gain vs LR : +0.063635


,model,auc,gini,logloss
0,LightGBM,0.765946,0.531892,0.290374
1,XGBoost,0.765593,0.531185,0.551099
2,Logistic Regression,0.701957,0.403915,0.605552


## Ensemble: LGBM + XGB

### Why combine models

Both models learn from the same data but with different tree construction strategies (leaf-wise vs level-wise). Their errors tend not to be perfectly correlated, so a weighted blend reduces variance and improves AUC.

Bryan used weights of 12% LGBM + 88% XGB, determined through 84 leaderboard submissions. These are empirically derived rather than theoretically optimal.

### Experimental results

Bryan 12/88 blend: AUC = 0.768082, improves by 0.007 over standalone LGBM.

Grid search (5% steps): optimal weights are LGBM 95% + XGB 5%, AUC = 0.769344.

Why our optimal weights differ from Bryan's:
- Bryan's XGB was strong (76 features, good convergence), LGBM weaker
- Our LGBM has better LogLoss calibration (0.298 vs 0.545 for XGB)
- The optimizer naturally weights the better-calibrated model higher

Once log features carry real values and LGBM convergence improves, the optimal blend will likely shift toward Bryan's 12/88 ratio.


In [25]:
# ===== 8. Ensemble =====

# ── Bryan's 12/88 blend ───────────────────────────────────────────────────────
LGBM_WEIGHT = 0.12
XGB_WEIGHT  = 0.88

ensemble_val_proba = LGBM_WEIGHT * lgbm_val_proba + XGB_WEIGHT * xgb_val_proba
ensemble_auc       = roc_auc_score(y_val, ensemble_val_proba)
ensemble_logloss   = log_loss(y_val, ensemble_val_proba)
ensemble_gini      = 2 * ensemble_auc - 1

print("=" * 55)
print(f"ENSEMBLE (LGBM {LGBM_WEIGHT*100:.0f}% + XGB {XGB_WEIGHT*100:.0f}%) — BRYAN'S SETUP")
print("=" * 55)
print(f"  Val AUC      : {ensemble_auc:.6f}")
print(f"  Val Gini     : {ensemble_gini:.6f}")
print(f"  Val LogLoss  : {ensemble_logloss:.6f}")
print(f"  vs LGBM alone: {ensemble_auc - lgbm_auc:+.6f}")
print(f"  vs XGB  alone: {ensemble_auc - xgb_auc:+.6f}")
print("=" * 55)

# ── Grid search best blend ratio ──────────────────────────────────────────────
print("\nSearching optimal blend ratio...")
best_w, best_blend_auc = LGBM_WEIGHT, ensemble_auc

for w_lgbm in np.arange(0.0, 1.01, 0.05):
    w_xgb  = 1.0 - w_lgbm
    proba  = w_lgbm * lgbm_val_proba + w_xgb * xgb_val_proba
    auc    = roc_auc_score(y_val, proba)
    if auc > best_blend_auc:
        best_blend_auc = auc
        best_w         = w_lgbm

print(f"  Optimal LGBM weight : {best_w:.2f}  (XGB: {1-best_w:.2f})")
print(f"  Optimal blend AUC   : {best_blend_auc:.6f}")
print(f"  Bryan's blend AUC   : {ensemble_auc:.6f}")
print(f"  Difference          : {best_blend_auc - ensemble_auc:+.6f}")

# ── Use best blend for final ensemble ────────────────────────────────────────
FINAL_LGBM_W = best_w
FINAL_XGB_W  = 1.0 - best_w
final_ensemble_val_proba = FINAL_LGBM_W * lgbm_val_proba + FINAL_XGB_W * xgb_val_proba
final_ensemble_auc       = roc_auc_score(y_val, final_ensemble_val_proba)

print(f"\nFinal ensemble weights: LGBM={FINAL_LGBM_W:.2f}, XGB={FINAL_XGB_W:.2f}")
print(f"Final ensemble AUC    : {final_ensemble_auc:.6f}")

ENSEMBLE_RESULTS = {
    "model": f"Ensemble (LGBM {FINAL_LGBM_W:.0%} + XGB {FINAL_XGB_W:.0%})",
    "auc": final_ensemble_auc,
    "gini": 2*final_ensemble_auc - 1,
    "logloss": log_loss(y_val, final_ensemble_val_proba),
    "time": 0,
}


ENSEMBLE (LGBM 12% + XGB 88%) — BRYAN'S SETUP
  Val AUC      : 0.765678
  Val Gini     : 0.531356
  Val LogLoss  : 0.491413
  vs LGBM alone: -0.000268
  vs XGB  alone: +0.000085

Searching optimal blend ratio...
  Optimal LGBM weight : 0.95  (XGB: 0.05)
  Optimal blend AUC   : 0.769059
  Bryan's blend AUC   : 0.765678
  Difference          : +0.003381

Final ensemble weights: LGBM=0.95, XGB=0.05
Final ensemble AUC    : 0.769059


## Feature Importance

Comparison of feature importance between LGBM and XGBoost based on gain — the average reduction in impurity when a feature is used to split a node.

Key observations from the current run:
- `registered_via` accounts for 46.3% in LGBM, 14.1% in XGBoost. This concentration in LGBM indicates the model is over-relying on a single feature, a symptom of other features (especially logs) carrying no signal.
- `days_last_txn_to_expire` ranks strongly in both (16.4% LGBM, 10.7% XGB), confirming it is a robust signal.
- `auto_renew_rate` and `last_is_auto_renew` contribute similarly in both models.
- No log features appear in the top 15 because all log values are zero in this run.

Expected change after re-running with the `num_uniq` fix: `registered_via` dominance should decrease and log features such as `days_since_last_login` and `total_secs_played` should appear — consistent with Bryan's top 10 where log features account for 7 of 10 positions.


In [26]:
# ===== 9. Feature Importance =====

# LGBM importance
lgbm_imp = pd.DataFrame({
    "feature"    : FEATURE_COLS,
    "lgbm_gain"  : lgbm_model.booster_.feature_importance(importance_type="gain"),
    "lgbm_split" : lgbm_model.booster_.feature_importance(importance_type="split"),
}).sort_values("lgbm_gain", ascending=False).reset_index(drop=True)

# XGB importance
xgb_imp_raw = xgb_model.get_booster().get_score(importance_type="gain")
xgb_imp = pd.DataFrame([
    {"feature": f, "xgb_gain": xgb_imp_raw.get(f, 0)} for f in FEATURE_COLS
]).sort_values("xgb_gain", ascending=False).reset_index(drop=True)

# Merge for comparison
imp_combined = lgbm_imp.merge(xgb_imp, on="feature", how="outer").fillna(0)
imp_combined["lgbm_gain_norm"] = (imp_combined["lgbm_gain"] /
                                   imp_combined["lgbm_gain"].sum() * 100)
imp_combined["xgb_gain_norm"]  = (imp_combined["xgb_gain"] /
                                   imp_combined["xgb_gain"].sum() * 100)
imp_combined = imp_combined.sort_values("lgbm_gain_norm", ascending=False)

print("=" * 65)
print("FEATURE IMPORTANCE (Gain %) — Top 15")
print("=" * 65)
top15 = imp_combined.head(15)[["feature","lgbm_gain_norm","xgb_gain_norm"]]
top15.columns = ["Feature", "LGBM Gain%", "XGB Gain%"]
print(top15.to_string(index=False))
print("=" * 65)

display(
    imp_combined.head(15)[["feature","lgbm_gain_norm","xgb_gain_norm"]]
    .rename(columns={"feature":"Feature","lgbm_gain_norm":"LGBM Gain%","xgb_gain_norm":"XGB Gain%"})
    .style.background_gradient(subset=["LGBM Gain%","XGB Gain%"], cmap="Blues")
    .format({"LGBM Gain%":"{:.2f}","XGB Gain%":"{:.2f}"})
    .hide(axis="index")
)


FEATURE IMPORTANCE (Gain %) — Top 15
                Feature  LGBM Gain%  XGB Gain%
         registered_via   42.011996  10.838480
  registration_date_abs   12.623717   2.453527
      total_amount_paid    9.946445   6.783800
         days_since_reg    7.286562   2.480566
     last_is_auto_renew    7.043419   3.910677
days_last_txn_to_expire    6.188851   4.684313
        auto_renew_rate    2.911909   7.380542
                     bd    2.446618   3.836342
             bd_missing    2.045856   4.203758
                 n_txns    1.171288   2.441707
 days_since_last_cancel    1.154105   2.636580
        avg_amount_paid    1.122636   1.906906
  recency_to_plan_ratio    0.997130   0.748649
                   city    0.893486   3.768266
          max_plan_days    0.533864   4.529331


Feature,LGBM Gain%,XGB Gain%
registered_via,42.01,10.84
registration_date_abs,12.62,2.45
total_amount_paid,9.95,6.78
days_since_reg,7.29,2.48
last_is_auto_renew,7.04,3.91
days_last_txn_to_expire,6.19,4.68
auto_renew_rate,2.91,7.38
bd,2.45,3.84
bd_missing,2.05,4.20
n_txns,1.17,2.44


## Step 11 — Inference, Calibration and Submission

Predictions are generated on 907,471 March 2017 users using the final ensemble.

### Why calibration is needed

The raw ensemble outputs probabilities anchored to the training distribution.
For a business retention system, we want predicted probabilities that reflect
**real-world churn rates as closely as possible**, so that:
- CLV calculations in the retention layer use accurate expected churn
- Voucher break-even thresholds are correctly derived
- Segment tier boundaries carry meaningful economic interpretation

The standard approach is **isotonic regression** calibration: fit a monotone
mapping from raw predictions to actual outcomes using the validation set, then
apply that mapping to inference predictions.

### What calibration does and does not change

- **AUC is preserved**: isotonic is a monotone transform — it cannot change
  the ranking of users, only the scale of their probabilities.
- **LogLoss improves**: calibrated probabilities are closer to the true 0/1
  outcomes, reducing the cross-entropy loss.
- **Business interpretation improves**: a prediction of 0.09 after calibration
  means "this user has approximately a 9% chance of churning" in a way that
  is consistent with the observed 8.99% validation churn rate.

### Note on narrow prediction range

The current inference shows mean=0.085, max=0.134, with 0% of users above 0.50.
This is a direct consequence of LGBM stopping at best_iteration=6 — the model
barely learned, producing predictions compressed near the population mean.
A retrained model (after the num_uniq FE fix) will produce a wider distribution
with genuinely high-risk users receiving probabilities well above 0.30.


In [27]:
# Fallback: use Bryan default weights if Cell 13 (grid search) was not run
if "FINAL_LGBM_W" not in dir():
    FINAL_LGBM_W = 0.12
    FINAL_XGB_W  = 0.88
    print("Warning: FINAL_LGBM_W not found — using Bryan default weights 12/88")

# ===== 10. Inference — March 2017 =====
print(f"Generating predictions for {X_inf.shape[0]:,} users...")
start_inf = time.time()

lgbm_inf_proba = lgbm_model.predict_proba(X_inf)[:, 1]
xgb_inf_proba  = xgb_model.predict_proba(X_inf)[:, 1]
ensemble_inf_proba = FINAL_LGBM_W * lgbm_inf_proba + FINAL_XGB_W * xgb_inf_proba

inf_time = time.time() - start_inf

print(f"  Inference time: {inf_time:.1f}s")
print(f"  Prediction stats:")
print(f"    Mean  : {ensemble_inf_proba.mean():.4f}")
print(f"    Median: {np.median(ensemble_inf_proba):.4f}")
print(f"    Std   : {ensemble_inf_proba.std():.4f}")
print(f"    Min   : {ensemble_inf_proba.min():.4f}")
print(f"    Max   : {ensemble_inf_proba.max():.4f}")
print(f"    % > 0.5: {(ensemble_inf_proba > 0.5).mean():.2%}")


Generating predictions for 907,471 users...
  Inference time: 0.3s
  Prediction stats:
    Mean  : 0.1010
    Median: 0.0928
    Std   : 0.0247
    Min   : 0.0706
    Max   : 0.1809
    % > 0.5: 0.00%


In [28]:
# 03-3 FIX: guard against oe (OrdinalEncoder) being undefined if kernel was cleared
# oe is fitted in Cell 03 (data preparation). If you restart the kernel,
# you must re-run from Cell 01 to Cell 17 before running this cell.
if "oe" not in dir():
    raise RuntimeError(
        "OrdinalEncoder 'oe' not found in scope. "
        "Please re-run from Cell 01 (imports) through Cell 17 (inference) first."
    )

# ===== 11. Calibration + Submission Export =====
import pickle
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, log_loss

# ── Fit isotonic calibration on validation set ────────────────────────────────
# Isotonic regression learns a monotone mapping from raw predictions to actual
# outcomes. Fitted on val set (Feb 2017 actual labels), applied to inference.
# This is the principled approach for a business system where we have no
# knowledge of the test set churn rate ahead of time.
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(final_ensemble_val_proba, y_val)

val_iso = iso.predict(final_ensemble_val_proba)

# ── Validate calibration on val set ──────────────────────────────────────────
val_churn_rate = float(y_val.mean())
print("=" * 58)
print("CALIBRATION VALIDATION (Feb 2017 val set)")
print("=" * 58)
print(f"{'Method':<20} {'Mean pred':>10} {'LogLoss':>10} {'AUC':>10}")
print("-" * 58)
for name, preds in [("Raw ensemble", final_ensemble_val_proba),
                    ("Isotonic",     val_iso)]:
    ll  = log_loss(y_val, preds)
    auc = roc_auc_score(y_val, preds)
    print(f"  {name:<18} {preds.mean():>10.4f} {ll:>10.6f} {auc:>10.6f}")

print(f"\n  Observed val churn rate : {val_churn_rate:.4f}")
print(f"  Raw mean prediction     : {final_ensemble_val_proba.mean():.4f}")
print(f"  Calibrated mean         : {val_iso.mean():.4f}")
print(f"  AUC unchanged after calibration: monotone transform preserves ranking")

# ── Apply calibration to inference predictions ────────────────────────────────
ensemble_inf_proba_calibrated = iso.predict(ensemble_inf_proba)

print()
print("=" * 58)
print("INFERENCE PREDICTIONS (March 2017 — 907,471 users)")
print("=" * 58)
for name, preds in [("Raw ensemble", ensemble_inf_proba),
                    ("Calibrated",   ensemble_inf_proba_calibrated)]:
    print(f"  {name:<20}: mean={preds.mean():.4f}  "
          f"median={np.median(preds):.4f}  "
          f"std={preds.std():.4f}  "
          f"max={preds.max():.4f}")

# ── Save submission ───────────────────────────────────────────────────────────
submission = pd.DataFrame({
    "msno"    : inf_df["msno"].values,
    "is_churn": ensemble_inf_proba_calibrated,
})
submission.to_csv(DATA_DIR / "submission.csv", index=False)
print(f"\nSaved: submission.csv ({len(submission):,} rows) — isotonic calibrated")

# Also save raw for comparison
pd.DataFrame({"msno": inf_df["msno"].values,
              "is_churn": ensemble_inf_proba}).to_csv(
    DATA_DIR / "submission_raw.csv", index=False)
pd.DataFrame({"msno": inf_df["msno"].values,
              "is_churn": lgbm_inf_proba}).to_csv(
    DATA_DIR / "submission_lgbm.csv", index=False)
pd.DataFrame({"msno": inf_df["msno"].values,
              "is_churn": xgb_inf_proba}).to_csv(
    DATA_DIR / "submission_xgb.csv", index=False)
print("Saved: submission_raw.csv, submission_lgbm.csv, submission_xgb.csv")

# ── Save model artifacts ──────────────────────────────────────────────────────
lgbm_model.booster_.save_model(str(MODELS_DIR / "lgbm_model.txt"))
xgb_model.save_model(str(MODELS_DIR / "xgb_model.json"))
with open(MODELS_DIR / "ordinal_encoder.pkl", "wb") as f:
    pickle.dump(oe, f)
print("Saved: lgbm_model.txt, xgb_model.json, ordinal_encoder.pkl")

# ── Save metadata ─────────────────────────────────────────────────────────────
model_meta = {
    "feature_cols"         : FEATURE_COLS,
    "cat_cols"             : CAT_COLS,
    "num_cols"             : NUM_COLS,
    "target"               : TARGET,
    "ensemble_lgbm_weight" : FINAL_LGBM_W,
    "ensemble_xgb_weight"  : FINAL_XGB_W,
    "calibration": {
        "method"           : "isotonic_regression",
        "fitted_on"        : "validation_set_Feb2017",
        "val_churn_rate"   : round(val_churn_rate, 6),
        "val_logloss_raw"  : round(log_loss(y_val, final_ensemble_val_proba), 6),
        "val_logloss_iso"  : round(log_loss(y_val, val_iso), 6),
        "val_auc"          : round(roc_auc_score(y_val, final_ensemble_val_proba), 6),
    },
    "val_results": {
        "logistic_regression": LR_RESULTS,
        "lightgbm"           : LGBM_RESULTS,
        "xgboost"            : XGB_RESULTS,
        "ensemble"           : ENSEMBLE_RESULTS,
    },
    "lgbm_best_params"  : lgbm_study.best_params,
    "xgb_best_params"   : xgb_study.best_params,
    "n_trials_lgbm"     : N_TRIALS_LGBM,
    "n_trials_xgb"      : N_TRIALS_XGB,
}
with open(DATA_DIR / "modeling_metadata.json", "w") as f:
    json.dump(model_meta, f, indent=2, default=str)
print("Saved: modeling_metadata.json")


CALIBRATION VALIDATION (Feb 2017 val set)
Method                Mean pred    LogLoss        AUC
----------------------------------------------------------
  Raw ensemble           0.1075   0.287733   0.769059
  Isotonic               0.0899   0.256835   0.771638

  Observed val churn rate : 0.0899
  Raw mean prediction     : 0.1075
  Calibrated mean         : 0.0899
  AUC unchanged after calibration: monotone transform preserves ranking

INFERENCE PREDICTIONS (March 2017 — 907,471 users)
  Raw ensemble        : mean=0.1010  median=0.0928  std=0.0247  max=0.1809
  Calibrated          : mean=0.0914  median=0.0396  std=0.1545  max=1.0000

Saved: submission.csv (907,471 rows) — isotonic calibrated
Saved: submission_raw.csv, submission_lgbm.csv, submission_xgb.csv
Saved: lgbm_model.txt, xgb_model.json, ordinal_encoder.pkl
Saved: modeling_metadata.json


In [29]:
# ===== 12. Final Summary =====
all_results = pd.DataFrame([LR_RESULTS, LGBM_RESULTS, XGB_RESULTS, ENSEMBLE_RESULTS])
all_results = all_results.sort_values("auc", ascending=False).reset_index(drop=True)

print("=" * 65)
print("FINAL LEADERBOARD — Validation AUC (Feb 2017)")
print("=" * 65)
for _, row in all_results.iterrows():
    bar = "█" * int(row["auc"] * 50)
    print(f"  {row['model']:<40} AUC: {row['auc']:.6f}")
print("=" * 65)
print(f"\nBest model   : {all_results.iloc[0]['model']}")
print(f"Best Val AUC : {all_results.iloc[0]['auc']:.6f}")
print(f"Best Val Gini: {all_results.iloc[0]['gini']:.6f}")
print(f"\nEnsemble vs LGBM alone : {ENSEMBLE_RESULTS['auc'] - LGBM_RESULTS['auc']:+.6f}")
print(f"Ensemble vs XGB  alone : {ENSEMBLE_RESULTS['auc'] - XGB_RESULTS['auc']:+.6f}")
print(f"\nSubmission files:")
print(f"  Data/submission.csv       ← Main (ensemble)")
print(f"  Data/submission_lgbm.csv  ← LGBM only")
print(f"  Data/submission_xgb.csv   ← XGB only")
print(f"\nEnsemble weights used: LGBM={FINAL_LGBM_W:.0%}, XGB={FINAL_XGB_W:.0%}")
print(f"Bryan's setup          : LGBM=12%, XGB=88%")

display(all_results[["model","auc","gini","logloss"]].style
    .background_gradient(subset=["auc","gini"], cmap="Greens")
    .background_gradient(subset=["logloss"], cmap="Reds_r")
    .format({"auc":"{:.6f}","gini":"{:.6f}","logloss":"{:.6f}"})
    .hide(axis="index"))


FINAL LEADERBOARD — Validation AUC (Feb 2017)
  Ensemble (LGBM 95% + XGB 5%)             AUC: 0.769059
  LightGBM                                 AUC: 0.765946
  XGBoost                                  AUC: 0.765593
  Logistic Regression                      AUC: 0.701957

Best model   : Ensemble (LGBM 95% + XGB 5%)
Best Val AUC : 0.769059
Best Val Gini: 0.538118

Ensemble vs LGBM alone : +0.003113
Ensemble vs XGB  alone : +0.003466

Submission files:
  Data/submission.csv       ← Main (ensemble)
  Data/submission_lgbm.csv  ← LGBM only
  Data/submission_xgb.csv   ← XGB only

Ensemble weights used: LGBM=95%, XGB=5%
Bryan's setup          : LGBM=12%, XGB=88%


model,auc,gini,logloss
Ensemble (LGBM 95% + XGB 5%),0.769059,0.538118,0.287733
LightGBM,0.765946,0.531892,0.290374
XGBoost,0.765593,0.531185,0.551099
Logistic Regression,0.701957,0.403915,0.605552
